In [1]:
%pip install -q duckdb

In [2]:
import duckdb

con = duckdb.connect("online_retail.duckdb")

print("DuckDB version:", duckdb.__version__)
print("Connection successful!")

DuckDB version: 1.3.2
Connection successful!


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

data_dir = Path("/content/drive/MyDrive/MVJ_Internship/Data")

print("Folder exists:", data_dir.exists())
print("\nCSV files found:")

for file in sorted(data_dir.glob("*.csv")):
    print(file.name, "|", round(file.stat().st_size / (1024 * 1024), 2), "MB")

Folder exists: True

CSV files found:
online_retail_II_csv(2009-2010).csv | 38.82 MB
online_retail_II_csv(2010-2011).csv | 39.38 MB


In [5]:
%pip install -q duckdb

In [6]:
import duckdb
from pathlib import Path

con = duckdb.connect("online_retail.duckdb")

csv_files = sorted(data_dir.glob("*.csv"))

if len(csv_files) < 2:
    raise FileNotFoundError(
        f"Expected at least 2 CSV files in {data_dir}, "
        f"but found {len(csv_files)}."
    )

for file in csv_files:
    print(f"\nFile: {file.name}")

    # Read the CSV directly without modifying the source file
    result = con.execute(
        "SELECT COUNT(*) FROM read_csv_auto(?, header=true)",
        [str(file)]
    ).fetchone()

    print("Data rows:", f"{result[0]:,}")


File: online_retail_II_csv(2009-2010).csv
Data rows: 525,461

File: online_retail_II_csv(2010-2011).csv
Data rows: 541,910


In [7]:
for file in csv_files:
    print(f"\n{'=' * 50}")
    print("FILE:", file.name)

    print("\nColumn names and data types:")
    print(
        con.execute(
            "DESCRIBE SELECT * FROM read_csv_auto(?, header=true)",
            [str(file)]
        ).fetchdf().to_string(index=False)
    )

    print("\nFirst 5 rows:")
    print(
        con.execute(
            "SELECT * FROM read_csv_auto(?, header=true) LIMIT 5",
            [str(file)]
        ).fetchdf().to_string(index=False)
    )


FILE: online_retail_II_csv(2009-2010).csv

Column names and data types:
column_name column_type null  key default extra
    Invoice     VARCHAR  YES None    None  None
  StockCode     VARCHAR  YES None    None  None
Description     VARCHAR  YES None    None  None
   Quantity      BIGINT  YES None    None  None
InvoiceDate        DATE  YES None    None  None
      Price      DOUBLE  YES None    None  None
Customer ID      BIGINT  YES None    None  None
    Country     VARCHAR  YES None    None  None

First 5 rows:
Invoice StockCode                         Description  Quantity InvoiceDate  Price  Customer ID        Country
 489434     85048 15CM CHRISTMAS GLASS BALL 20 LIGHTS        12  2001-12-09   6.95        13085 United Kingdom
 489434    79323P                  PINK CHERRY LIGHTS        12  2001-12-09   6.75        13085 United Kingdom
 489434    79323W                 WHITE CHERRY LIGHTS        12  2001-12-09   6.75        13085 United Kingdom
 489434     22041        RECORD FRAM

In [26]:
#combine both csv files into one duckDB view
from pathlib import Path

assert len(csv_files) == 2, (
    f"Expected exactly 2 CSV files, found {len(csv_files)}: "
    f"{[f.name for f in csv_files]}"
)

# Confirm that both files have the same column names
column_lists = []

for file in csv_files:
    columns = con.execute(
        "DESCRIBE SELECT * FROM read_csv_auto(?, header=true, ignore_errors=true)",
        [str(file)]
    ).fetchdf()["column_name"].tolist()

    column_lists.append(columns)

    print(file.name, "->", columns)

if column_lists[0] != column_lists[1]:
    raise ValueError(
        "The two CSV files have different column names or column order. "
        "Stop here and share the output above."
    )

# Create a view over both files
file_a = csv_files[0].as_posix()
file_b = csv_files[1].as_posix()

# Using f-strings because DDL queries (CREATE VIEW) cannot accept prepared parameters
con.execute(f"""
    CREATE OR REPLACE VIEW online_retail AS
    SELECT * FROM read_csv_auto('{file_a}', header=true, ignore_errors=true)
    UNION ALL
    SELECT * FROM read_csv_auto('{file_b}', header=true, ignore_errors=true)
""")

print("Combined view created successfully with ignore_errors=true.")

online_retail_II_csv(2009-2010).csv -> ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
online_retail_II_csv(2010-2011).csv -> ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Combined view created successfully with ignore_errors=true.


In [9]:
#verify the combined dataset
print("Combined data rows:")
print(f"{con.execute('SELECT COUNT(*) FROM online_retail').fetchone()[0]:,}")

print("\nColumn names and types:")
print(
    con.execute("DESCRIBE online_retail")
    .fetchdf()
    .to_string(index=False)
)

print("\nSample records:")
print(
    con.execute("SELECT * FROM online_retail LIMIT 5")
    .fetchdf()
    .to_string(index=False)
)

Combined data rows:
1,067,371

Column names and types:
column_name column_type null  key default extra
    Invoice     VARCHAR  YES None    None  None
  StockCode     VARCHAR  YES None    None  None
Description     VARCHAR  YES None    None  None
   Quantity      BIGINT  YES None    None  None
InvoiceDate        DATE  YES None    None  None
      Price      DOUBLE  YES None    None  None
Customer ID      BIGINT  YES None    None  None
    Country     VARCHAR  YES None    None  None

Sample records:
Invoice StockCode                         Description  Quantity InvoiceDate  Price  Customer ID        Country
 489434     85048 15CM CHRISTMAS GLASS BALL 20 LIGHTS        12  2001-12-09   6.95        13085 United Kingdom
 489434    79323P                  PINK CHERRY LIGHTS        12  2001-12-09   6.75        13085 United Kingdom
 489434    79323W                 WHITE CHERRY LIGHTS        12  2001-12-09   6.75        13085 United Kingdom
 489434     22041        RECORD FRAME 7" SINGLE SIZE

In [10]:
#run basuc data quality checks
quality_sql = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE InvoiceDate IS NULL
    ) AS missing_dates,
    COUNT(*) FILTER (
        WHERE Quantity IS NULL
    ) AS missing_quantities,
    COUNT(*) FILTER (
        WHERE Price IS NULL
    ) AS missing_prices,
    COUNT(*) FILTER (
        WHERE "Customer ID" IS NULL
    ) AS missing_customer_ids,
    COUNT(DISTINCT Invoice) AS distinct_invoices,
    MIN(InvoiceDate) AS first_transaction,
    MAX(InvoiceDate) AS last_transaction
FROM online_retail
"""

quality = con.execute(quality_sql).fetchdf()
display(quality)

print("Cancellation rows:")
print(con.execute("""
    SELECT COUNT(*)
    FROM online_retail
    WHERE starts_with(upper(trim(CAST(Invoice AS VARCHAR))), 'C')
""").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_dates,missing_quantities,missing_prices,missing_customer_ids,distinct_invoices,first_transaction,last_transaction
0,1067371,0,0,0,243007,53628,2001-02-10,2031-10-10


Cancellation rows:
19494


In [11]:
#create a consistent analysis dataset
con.execute("""
    CREATE OR REPLACE VIEW retail_sales AS
    SELECT
        *,
        CAST(Quantity AS DOUBLE) * CAST(Price AS DOUBLE) AS Revenue
    FROM online_retail
    WHERE InvoiceDate IS NOT NULL
      AND Quantity > 0
      AND Price > 0
      AND NOT starts_with(
          upper(trim(CAST(Invoice AS VARCHAR))), 'C'
      )
""")

print("Original rows:", f"{con.execute('SELECT COUNT(*) FROM online_retail').fetchone()[0]:,}")
print("Rows retained for sales analysis:", f"{con.execute('SELECT COUNT(*) FROM retail_sales').fetchone()[0]:,}")
print("Excluded rows:", f"{con.execute('SELECT COUNT(*) FROM online_retail').fetchone()[0] - con.execute('SELECT COUNT(*) FROM retail_sales').fetchone()[0]:,}")


Original rows: 1,067,371
Rows retained for sales analysis: 1,041,670
Excluded rows: 25,701


Day 5 **Deliverables**

In [12]:
monthly_sql = """
SELECT
    strftime(InvoiceDate, '%Y-%m') AS sales_month,
    COUNT(DISTINCT Invoice) AS total_orders,
    ROUND(SUM(Revenue), 2) AS total_revenue,
    ROUND(
        SUM(Revenue) / NULLIF(COUNT(DISTINCT Invoice), 0),
        2
    ) AS average_order_value
FROM retail_sales
GROUP BY sales_month
ORDER BY sales_month
"""

monthly_kpis = con.execute(monthly_sql).fetchdf()
display(monthly_kpis)

,sales_month,total_orders,total_revenue,average_order_value
0,2001-02,66,19831.43,300.48
1,2001-03,58,25831.21,445.37
2,2001-04,79,30823.15,390.17
3,2001-06,47,15732.15,334.73
4,2001-07,85,30314.81,356.64
...,...,...,...,...
295,2030-11,123,60029.08,488.04
296,2031-01,53,18704.68,352.92
297,2031-03,75,29718.33,396.24
298,2031-08,63,35964.47,570.86


In [13]:
#country wise performance
country_sql = """
SELECT
    Country,
    COUNT(DISTINCT Invoice) AS total_orders,
    ROUND(SUM(Revenue), 2) AS total_revenue,
    ROUND(
        SUM(Revenue) / NULLIF(COUNT(DISTINCT Invoice), 0),
        2
    ) AS average_order_value
FROM retail_sales
GROUP BY Country
ORDER BY total_revenue DESC
"""

country_kpis = con.execute(country_sql).fetchdf()
display(country_kpis)

,Country,total_orders,total_revenue,average_order_value
0,United Kingdom,36535,17870977.78,489.15
1,EIRE,626,664431.78,1061.39
2,Netherlands,228,554232.34,2430.84
3,Germany,789,431262.46,546.59
4,France,622,356944.60,573.87
5,Australia,95,169968.11,1789.14
6,Spain,154,109178.53,708.95
7,Switzerland,93,101011.29,1086.14
8,Sweden,105,91903.72,875.27
9,Denmark,43,69862.19,1624.70


In [14]:
#monthly performance by country
monthly_country_sql = """
SELECT
    strftime(InvoiceDate, '%Y-%m') AS sales_month,
    Country,
    COUNT(DISTINCT Invoice) AS total_orders,
    ROUND(SUM(Revenue), 2) AS total_revenue,
    ROUND(
        SUM(Revenue) / NULLIF(COUNT(DISTINCT Invoice), 0),
        2
    ) AS average_order_value
FROM retail_sales
GROUP BY sales_month, Country
ORDER BY sales_month, total_revenue DESC
"""

monthly_country_kpis = con.execute(monthly_country_sql).fetchdf()
display(monthly_country_kpis)

,sales_month,Country,total_orders,total_revenue,average_order_value
0,2001-02,United Kingdom,57,16315.82,286.24
1,2001-02,Malta,2,1636.38,818.19
2,2001-02,Germany,2,824.20,412.10
3,2001-02,Spain,2,492.04,246.02
4,2001-02,Portugal,2,403.33,201.67
...,...,...,...,...,...
1623,2031-08,France,2,251.02,125.51
1624,2031-08,Netherlands,1,147.09,147.09
1625,2031-10,United Kingdom,80,29245.82,365.57
1626,2031-10,Iceland,1,611.53,611.53


In [15]:
#export the results to drive
from pathlib import Path

output_dir = Path(
    "/content/drive/MyDrive/MVJ_Internship/Project1/sql"
)
output_dir.mkdir(parents=True, exist_ok=True)

monthly_kpis.to_csv(output_dir / "monthly_kpis.csv", index=False)
country_kpis.to_csv(output_dir / "country_kpis.csv", index=False)
monthly_country_kpis.to_csv(
    output_dir / "monthly_country_kpis.csv",
    index=False
)

print("Exported files:")
for file in sorted(output_dir.glob("*.csv")):
    print(file.name, "-", file.stat().st_size, "bytes")

Exported files:
country_kpis.csv - 1254 bytes
customer_level.csv - 290355 bytes
customer_segment_summary.csv - 227 bytes
customer_segments.csv - 356053 bytes
monthly_country_kpis.csv - 55496 bytes
monthly_kpis.csv - 8175 bytes


In [16]:
#save the sql queries
queries_sql = """-- MVJ Internship: Day 5 SQL Aggregation Analysis
-- Dataset: UCI Online Retail II
-- Initial sales filter excludes cancellations, non-positive quantity/price,
-- and missing invoice dates.

-- 1. Data quality
""" + quality_sql + """;

-- 2. Monthly KPIs
""" + monthly_sql + """;

-- 3. Country KPIs
""" + country_sql + """;

-- 4. Monthly country KPIs
""" + monthly_country_sql + """;
"""

(output_dir / "queries.sql").write_text(
    queries_sql,
    encoding="utf-8"
)

print("Saved:", output_dir / "queries.sql")

Saved: /content/drive/MyDrive/MVJ_Internship/Project1/sql/queries.sql


In [17]:
#identify 3 real findings
print("TOP 5 COUNTRIES BY REVENUE")
display(country_kpis.head(5))

print("TOP 5 MONTHS BY REVENUE")
display(
    monthly_kpis.sort_values(
        "total_revenue", ascending=False
    ).head(5)
)

print("MONTH WITH THE MOST ORDERS")
display(
    monthly_kpis.sort_values(
        "total_orders", ascending=False
    ).head(1)
)

TOP 5 COUNTRIES BY REVENUE


,Country,total_orders,total_revenue,average_order_value
0,United Kingdom,36535,17870977.78,489.15
1,EIRE,626,664431.78,1061.39
2,Netherlands,228,554232.34,2430.84
3,Germany,789,431262.46,546.59
4,France,622,356944.60,573.87


TOP 5 MONTHS BY REVENUE


,sales_month,total_orders,total_revenue,average_order_value
108,2011-11,2898,1573737.31,543.04
107,2011-10,2131,1198489.17,562.41
106,2011-09,1837,1058590.17,576.26
97,2010-12,1651,868188.25,525.86
103,2011-06,1619,799542.15,493.85


MONTH WITH THE MOST ORDERS


,sales_month,total_orders,total_revenue,average_order_value
108,2011-11,2898,1573737.31,543.04


In [18]:
print(con.execute("SHOW TABLES").fetchdf())

            name
0  online_retail
1   retail_sales


In [19]:
print(con.execute("""
    SELECT
        COUNT(*) AS sales_rows,
        MIN(InvoiceDate) AS first_date,
        MAX(InvoiceDate) AS last_date
    FROM retail_sales
""").fetchdf())

   sales_rows first_date  last_date
0     1041670 2001-02-10 2031-10-10


In [20]:
#window functions
window_practice = con.execute("""
    SELECT
        sales_month,
        monthly_revenue,
        SUM(monthly_revenue) OVER (
            ORDER BY sales_month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_revenue,
        LAG(monthly_revenue) OVER (
            ORDER BY sales_month
        ) AS previous_month_revenue
    FROM (
        SELECT
            DATE_TRUNC('month', InvoiceDate)::DATE AS sales_month,
            SUM(Revenue) AS monthly_revenue
        FROM retail_sales
        GROUP BY 1
    ) AS monthly
    ORDER BY sales_month
""").fetchdf()

display(window_practice)

,sales_month,monthly_revenue,running_revenue,previous_month_revenue
0,2001-02-01,19831.431,1.983143e+04,NaN
1,2001-03-01,25831.210,4.566264e+04,19831.431
2,2001-04-01,30823.150,7.648579e+04,25831.210
3,2001-06-01,15732.150,9.221794e+04,30823.150
4,2001-07-01,30314.810,1.225328e+05,15732.150
...,...,...,...,...
295,2030-11-01,60029.080,2.085785e+07,51641.940
296,2031-01-01,18704.680,2.087655e+07,60029.080
297,2031-03-01,29718.330,2.090627e+07,18704.680
298,2031-08-01,35964.470,2.094224e+07,29718.330


In [21]:
#calculate monthly revenue and running totals
monthly_window_sql = """
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', InvoiceDate)::DATE AS sales_month,
        COUNT(DISTINCT Invoice) AS total_orders,
        SUM(Revenue) AS monthly_revenue
    FROM retail_sales
    GROUP BY 1
)
SELECT
    sales_month,
    total_orders,
    ROUND(monthly_revenue, 2) AS monthly_revenue,
    ROUND(
        monthly_revenue / NULLIF(total_orders, 0),
        2
    ) AS average_order_value,
    ROUND(
        SUM(monthly_revenue) OVER (
            ORDER BY sales_month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS running_revenue
FROM monthly
ORDER BY sales_month
"""

monthly_window = con.execute(monthly_window_sql).fetchdf()
display(monthly_window)

,sales_month,total_orders,monthly_revenue,average_order_value,running_revenue
0,2001-02-01,66,19831.43,300.48,19831.43
1,2001-03-01,58,25831.21,445.37,45662.64
2,2001-04-01,79,30823.15,390.17,76485.79
3,2001-06-01,47,15732.15,334.73,92217.94
4,2001-07-01,85,30314.81,356.64,122532.75
...,...,...,...,...,...
295,2030-11-01,123,60029.08,488.04,20857848.39
296,2031-01-01,53,18704.68,352.92,20876553.07
297,2031-03-01,75,29718.33,396.24,20906271.40
298,2031-08-01,63,35964.47,570.86,20942235.87


In [22]:
#calculate month-on-month growth with LAG
mom_growth_sql = """
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', InvoiceDate)::DATE AS sales_month,
        SUM(Revenue) AS monthly_revenue
    FROM retail_sales
    GROUP BY 1
),
with_previous AS (
    SELECT
        sales_month,
        monthly_revenue,
        LAG(monthly_revenue) OVER (
            ORDER BY sales_month
        ) AS previous_month_revenue
    FROM monthly
)
SELECT
    sales_month,
    ROUND(monthly_revenue, 2) AS monthly_revenue,
    ROUND(previous_month_revenue, 2) AS previous_month_revenue,
    ROUND(monthly_revenue - previous_month_revenue, 2)
        AS revenue_change,
    ROUND(
        100.0 * (monthly_revenue - previous_month_revenue)
        / NULLIF(previous_month_revenue, 0),
        2
    ) AS mom_growth_pct
FROM with_previous
ORDER BY sales_month
"""

mom_growth = con.execute(mom_growth_sql).fetchdf()
display(mom_growth)

,sales_month,monthly_revenue,previous_month_revenue,revenue_change,mom_growth_pct
0,2001-02-01,19831.43,NaN,NaN,NaN
1,2001-03-01,25831.21,19831.43,5999.78,30.25
2,2001-04-01,30823.15,25831.21,4991.94,19.33
3,2001-06-01,15732.15,30823.15,-15091.00,-48.96
4,2001-07-01,30314.81,15732.15,14582.66,92.69
...,...,...,...,...,...
295,2030-11-01,60029.08,51641.94,8387.14,16.24
296,2031-01-01,18704.68,60029.08,-41324.40,-68.84
297,2031-03-01,29718.33,18704.68,11013.65,58.88
298,2031-08-01,35964.47,29718.33,6246.14,21.02


In [27]:
top_products_sql = """
WITH product_monthly AS (
    SELECT
        DATE_TRUNC('month', InvoiceDate)::DATE AS sales_month,
        COALESCE(
            NULLIF(TRIM(CAST(Description AS VARCHAR)), ''),
            'Unknown product'
        ) AS product_name,
        SUM(Revenue) AS product_revenue,
        SUM(Quantity) AS units_sold
    FROM retail_sales
    GROUP BY 1, 2
),
ranked AS (
    SELECT
        sales_month,
        product_name,
        ROUND(product_revenue, 2) AS product_revenue,
        units_sold,
        ROW_NUMBER() OVER (
            PARTITION BY sales_month
            ORDER BY product_revenue DESC, product_name
        ) AS revenue_rank
    FROM product_monthly
)
SELECT *
FROM ranked
WHERE revenue_rank <= 5
ORDER BY sales_month, revenue_rank
"""

top_products = con.execute(top_products_sql).fetchdf()
display(top_products)

,sales_month,product_name,product_revenue,units_sold,revenue_rank
0,2001-02-01,Manual,818.19,1.0,1
1,2001-02-01,COOK WITH WINE METAL SIGN,800.28,468.0,2
2,2001-02-01,DOTCOM POSTAGE,573.33,9.0,3
3,2001-02-01,BLACK RECORD COVER FRAME,370.44,108.0,4
4,2001-02-01,WHITE HANGING HEART T-LIGHT HOLDER,305.82,106.0,5
...,...,...,...,...,...
1495,2031-10-01,REGENCY CAKESTAND 3 TIER,694.65,59.0,1
1496,2031-10-01,DOORMAT HOME SWEET HOME BLUE,571.80,84.0,2
1497,2031-10-01,DOORMAT MERRY CHRISTMAS RED,472.50,70.0,3
1498,2031-10-01,WHITE HANGING HEART T-LIGHT HOLDER,450.35,157.0,4


In [28]:
#identify each customer's first purchase month
first_purchase_sql = """
WITH customer_first_purchase AS (
    SELECT
        TRY_CAST("Customer ID" AS BIGINT) AS customer_id,
        MIN(InvoiceDate) AS first_purchase_date
    FROM retail_sales
    WHERE TRY_CAST("Customer ID" AS BIGINT) IS NOT NULL
    GROUP BY 1
)
SELECT
    customer_id,
    first_purchase_date,
    DATE_TRUNC('month', first_purchase_date)::DATE
        AS first_purchase_month
FROM customer_first_purchase
ORDER BY first_purchase_date, customer_id
"""

first_purchases = con.execute(first_purchase_sql).fetchdf()
display(first_purchases.head(20))

,customer_id,first_purchase_date,first_purchase_month
0,12540,2001-02-10,2001-02-01
1,12647,2001-02-10,2001-02-01
2,12711,2001-02-10,2001-02-01
3,12800,2001-02-10,2001-02-01
4,12810,2001-02-10,2001-02-01
5,13156,2001-02-10,2001-02-01
6,13173,2001-02-10,2001-02-01
7,13206,2001-02-10,2001-02-01
8,13259,2001-02-10,2001-02-01
9,13510,2001-02-10,2001-02-01


In [29]:
#build a monthly customer cohort summary
cohort_summary_sql = """
WITH customer_first_purchase AS (
    SELECT
        TRY_CAST("Customer ID" AS BIGINT) AS customer_id,
        DATE_TRUNC('month', MIN(InvoiceDate))::DATE
            AS cohort_month
    FROM retail_sales
    WHERE TRY_CAST("Customer ID" AS BIGINT) IS NOT NULL
    GROUP BY 1
)
SELECT
    cohort_month,
    COUNT(*) AS new_customers
FROM customer_first_purchase
GROUP BY cohort_month
ORDER BY cohort_month
"""

cohort_summary = con.execute(cohort_summary_sql).fetchdf()
display(cohort_summary)

,cohort_month,new_customers
0,2001-02-01,49
1,2001-03-01,42
2,2001-04-01,69
3,2001-06-01,31
4,2001-07-01,73
...,...,...
285,2030-11-01,3
286,2031-01-01,5
287,2031-03-01,4
288,2031-08-01,3


In [30]:
#calculate no.of months bw each customer's first purchase and each subsequent active month
cohort_activity_sql = """
WITH customer_cohorts AS (
    SELECT
        TRY_CAST("Customer ID" AS BIGINT) AS customer_id,
        DATE_TRUNC('month', MIN(InvoiceDate))::DATE AS cohort_month
    FROM retail_sales
    WHERE TRY_CAST("Customer ID" AS BIGINT) IS NOT NULL
    GROUP BY 1
),
customer_active_months AS (
    SELECT DISTINCT
        TRY_CAST("Customer ID" AS BIGINT) AS customer_id,
        DATE_TRUNC('month', InvoiceDate)::DATE AS active_month
    FROM retail_sales
    WHERE TRY_CAST("Customer ID" AS BIGINT) IS NOT NULL
)
SELECT
    c.cohort_month,
    a.active_month,
    DATE_DIFF('month', c.cohort_month, a.active_month)
        AS months_since_first_purchase,
    COUNT(DISTINCT c.customer_id) AS active_customers
FROM customer_cohorts c
JOIN customer_active_months a
    ON c.customer_id = a.customer_id
GROUP BY 1, 2, 3
ORDER BY 1, 3
"""

cohort_activity = con.execute(cohort_activity_sql).fetchdf()
display(cohort_activity)

,cohort_month,active_month,months_since_first_purchase,active_customers
0,2001-02-01,2001-02-01,0,49
1,2001-02-01,2001-03-01,1,7
2,2001-02-01,2001-04-01,2,3
3,2001-02-01,2001-07-01,5,2
4,2001-02-01,2001-08-01,6,1
...,...,...,...,...
9743,2030-11-01,2030-11-01,0,3
9744,2031-01-01,2031-01-01,0,5
9745,2031-03-01,2031-03-01,0,4
9746,2031-08-01,2031-08-01,0,3


In [31]:
#validate the results
print("Monthly revenue and running total")
display(con.execute("""
    WITH m AS (
        SELECT
            DATE_TRUNC('month', InvoiceDate)::DATE AS sales_month,
            SUM(Revenue) AS revenue
        FROM retail_sales
        GROUP BY 1
    )
    SELECT
        sales_month,
        revenue,
        SUM(revenue) OVER (ORDER BY sales_month) AS running_revenue
    FROM m
    ORDER BY sales_month
""").fetchdf())

print("Top-N validation: number of products per month")
display(con.execute("""
    SELECT sales_month, COUNT(*) AS products_returned,
           MAX(revenue_rank) AS highest_rank
    FROM (
        """ + top_products_sql + """
    ) t
    GROUP BY sales_month
    ORDER BY sales_month
""").fetchdf())

print("Customer cohort validation")
print(
    "Unique customers in first-purchase results:",
    first_purchases["customer_id"].nunique()
)
print(
    "Rows in first-purchase results:",
    len(first_purchases)
)

Monthly revenue and running total


,sales_month,revenue,running_revenue
0,2001-02-01,19831.431,1.983143e+04
1,2001-03-01,25831.210,4.566264e+04
2,2001-04-01,30823.150,7.648579e+04
3,2001-06-01,15732.150,9.221794e+04
4,2001-07-01,30314.810,1.225328e+05
...,...,...,...
295,2030-11-01,60029.080,2.085785e+07
296,2031-01-01,18704.680,2.087655e+07
297,2031-03-01,29718.330,2.090627e+07
298,2031-08-01,35964.470,2.094224e+07


Top-N validation: number of products per month


,sales_month,products_returned,highest_rank
0,2001-02-01,5,5
1,2001-03-01,5,5
2,2001-04-01,5,5
3,2001-06-01,5,5
4,2001-07-01,5,5
...,...,...,...
295,2030-11-01,5,5
296,2031-01-01,5,5
297,2031-03-01,5,5
298,2031-08-01,5,5


Customer cohort validation
Unique customers in first-purchase results: 5878
Rows in first-purchase results: 5878


In [32]:
#export all outputs to google drive
from pathlib import Path

day7_dir = Path(
    "/content/drive/MyDrive/MVJ_Internship/Project1/sql/day7"
)
day7_dir.mkdir(parents=True, exist_ok=True)

outputs = {
    "monthly_running_totals.csv": monthly_window,
    "mom_growth.csv": mom_growth,
    "top_5_products_per_month.csv": top_products,
    "first_purchase_months.csv": first_purchases,
    "cohort_summary.csv": cohort_summary,
    "cohort_activity.csv": cohort_activity,
}

for filename, dataframe in outputs.items():
    filepath = day7_dir / filename
    dataframe.to_csv(filepath, index=False)
    print(f"Saved {filename}: {len(dataframe):,} rows")

Saved monthly_running_totals.csv: 300 rows
Saved mom_growth.csv: 300 rows
Saved top_5_products_per_month.csv: 1,500 rows
Saved first_purchase_months.csv: 5,878 rows
Saved cohort_summary.csv: 290 rows
Saved cohort_activity.csv: 9,748 rows


In [33]:
#save the sql queries as a single file
day7_sql = """-- MVJ Internship - Day 7
-- SQL 3: Window functions for analytics
-- Dataset: UCI Online Retail II
-- Prerequisite: retail_sales view created in Day 5.

-- 1. Monthly running revenue
""" + monthly_window_sql + """;

-- 2. Month-on-month revenue growth with LAG
""" + mom_growth_sql + """;

-- 3. Top 5 products by revenue within each month
""" + top_products_sql + """;

-- 4. First purchase month per identified customer
""" + first_purchase_sql + """;

-- 5. First-purchase cohort summary
""" + cohort_summary_sql + """;

-- 6. Cohort activity by months since first purchase
""" + cohort_activity_sql + """;
"""

sql_file = day7_dir / "day7_window_functions.sql"
sql_file.write_text(day7_sql, encoding="utf-8")

print("SQL file saved:", sql_file)

SQL file saved: /content/drive/MyDrive/MVJ_Internship/Project1/sql/day7/day7_window_functions.sql


In [34]:
#display the top 3 findings from the found results
print("Highest-revenue month:")
display(
    monthly_window.sort_values(
        "monthly_revenue", ascending=False
    ).head(1)
)

print("Largest month-on-month growth:")
display(
    mom_growth.dropna(subset=["mom_growth_pct"])
    .sort_values("mom_growth_pct", ascending=False)
    .head(1)
)

print("Largest first-purchase cohort:")
display(
    cohort_summary.sort_values(
        "new_customers", ascending=False
    ).head(1)
)

Highest-revenue month:


,sales_month,total_orders,monthly_revenue,average_order_value,running_revenue
108,2011-11-01,2898,1573737.31,543.04,13996537.52


Largest month-on-month growth:


,sales_month,monthly_revenue,previous_month_revenue,revenue_change,mom_growth_pct
97,2010-12-01,868188.25,80244.63,787943.62,981.93


Largest first-purchase cohort:


,cohort_month,new_customers
100,2011-03-01,282
